In [1]:
import hail as hl
#import argparse
#import pandas
import os

from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc
from ko_utils import analysis

In [2]:
import hail as hl
from gnomad.utils.vep import process_consequences

In [3]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe086.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_test_export.log


In [58]:
#mt1 = qc.get_table('data/unphased/post-qc/ukb_wes_200k_filtered_chr21.mt', 'mt')
#mt2 = qc.get_table('data/phased/ukb_wes_phased_non_singleton_chr21-24xlong.qc-v4.2.2.vcf.gz', 'vcf')
input_path = 'data/phased/ukb_wes_phased_non_singleton_chr21-24xlong.qc-v4.2.2.vcf.gz'
mt = hl.import_vcf(input_path, force_bgz=True, array_elements_required=True, min_partitions=100)

In [ ]:
mt.show()

2021-10-08 13:04:29 Hail: INFO: Coerced prefix-sorted dataset


In [34]:
#mt2 = qc.filter_to_european(mt2, genetically_european = False, only_annotate = False)

def test(mt, genetically_european = True, only_annotate = False):
    r'''Get white british (app 11867) /well/lindgren/UKBIOBANK/DATA/QC/ukb_sqc_v2.txt
    or genetically european by projecting ancestries into 1KG prpject data''' 
    if genetically_european:
        ht1 = hl.import_table('/well/lindgren/dpalmer/ukb_get_EUR/data/final_EUR_list.tsv', no_header = True).rename({'f0':'eid'}).key_by('eid')
        ht1 = ht1.annotate(eur = 1)
        mt = mt.annotate_cols(eur = ht1[mt.s].eur)
    else:
        ht2 = hl.import_table('/well/lindgren/flassen/ressources/ukb/white_british/210921_ukbb_white_british_samples.txt',
            types={'eid': hl.tstr, 'in.white.British.ancestry.subset': hl.tint32}).key_by('eid')
        mt = mt.annotate_cols(eur = ht2[mt.s]['in.white.British.ancestry.subset'])
    # count and subset
    undefined_eur = mt.aggregate_cols(hl.agg.sum(hl.is_missing(mt.eur)))
    pre_filter_count = mt.count()
    if undefined_eur == pre_filter_count[1]:
        raise ValueError('[get_european]: IDs for europeans does not match keys in MatrixTable!')
    if undefined_eur > 0:
        print(f'[get_european]: Not all samples IDs mapped perfectly ({undefined_eur}/{pre_filter_count[1]} IDs are undefined)')
    if only_annotate == False:
        mt = mt.filter_cols(mt.eur == 1)
        post_filter_count = mt.count()
        print(f'[get_european]:{post_filter_count[1]}/{pre_filter_count[1]} IDs were included as genetically european.')
    return mt





In [44]:
mt = mt2
ht1 = hl.import_table('/well/lindgren/dpalmer/ukb_get_EUR/data/final_EUR_list.tsv', no_header = True).rename({'f0':'s'}).key_by('s')
ht1 = ht1.annotate(eur = 1)

mt.s.show()

#ht1[mt.s].eur.show()
#mt.show()




2021-10-08 12:51:30 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
2021-10-08 12:51:55 Hail: INFO: Coerced prefix-sorted dataset


FatalError: IllegalArgumentException: requirement failed

Java stack trace:
java.lang.IllegalArgumentException: requirement failed
	at scala.Predef$.require(Predef.scala:268)
	at is.hail.rvd.RVDPartitioner.<init>(RVDPartitioner.scala:52)
	at is.hail.rvd.RVDPartitioner.extendKeySamePartitions(RVDPartitioner.scala:141)
	at is.hail.expr.ir.LoweredTableReader$$anon$2.coerce(TableIR.scala:387)
	at is.hail.expr.ir.GenericTableValue.toTableStage(GenericTableValue.scala:159)
	at is.hail.io.vcf.MatrixVCFReader.lower(LoadVCF.scala:1791)
	at is.hail.expr.ir.lowering.LowerTableIR$.lower$1(LowerTableIR.scala:402)
	at is.hail.expr.ir.lowering.LowerTableIR$.apply(LowerTableIR.scala:1192)
	at is.hail.expr.ir.lowering.LowerToCDA$.lower(LowerToCDA.scala:68)
	at is.hail.expr.ir.lowering.LowerToCDA$.apply(LowerToCDA.scala:17)
	at is.hail.expr.ir.lowering.LowerToDistributedArrayPass.transform(LoweringPass.scala:76)
	at is.hail.expr.ir.LowerOrInterpretNonCompilable$.evaluate$1(LowerOrInterpretNonCompilable.

2021-10-08 12:52:19 Hail: INFO: Coerced prefix-sorted dataset


FatalError: IllegalArgumentException: requirement failed

Java stack trace:
java.lang.IllegalArgumentException: requirement failed
	at scala.Predef$.require(Predef.scala:268)
	at is.hail.rvd.RVDPartitioner.<init>(RVDPartitioner.scala:52)
	at is.hail.rvd.RVDPartitioner.extendKeySamePartitions(RVDPartitioner.scala:141)
	at is.hail.expr.ir.LoweredTableReader$$anon$2.coerce(TableIR.scala:387)
	at is.hail.expr.ir.GenericTableValue.toTableStage(GenericTableValue.scala:159)
	at is.hail.io.vcf.MatrixVCFReader.lower(LoadVCF.scala:1791)
	at is.hail.expr.ir.lowering.LowerTableIR$.lower$1(LowerTableIR.scala:402)
	at is.hail.expr.ir.lowering.LowerTableIR$.apply(LowerTableIR.scala:1192)
	at is.hail.expr.ir.lowering.LowerToCDA$.lower(LowerToCDA.scala:68)
	at is.hail.expr.ir.lowering.LowerToCDA$.apply(LowerToCDA.scala:17)
	at is.hail.expr.ir.lowering.LowerToDistributedArrayPass.transform(LoweringPass.scala:76)
	at is.hail.expr.ir.LowerOrInterpretNonCompilable$.evaluate$1(LowerOrInterpretNonCompilable.

In [12]:
#mt1 = qc.get_table('data/tmp/ukb_wes_200k_phased_tmp_chr22.1of1.vcf.gz', 'vcf')
#mt1 = qc.get_table('data/tmp/ukb_wes_200k_phased_tmp_chr22.1of1.vcf.gz', 'vcf')
#mt1 = qc.recalc_info(mt1)
#mt1 = qc.translate_sample_ids(mt1, from_app='12788', to_app='11867') # translate to lindgren app ID
#mt1 = qc.filter_to_european(mt1)
#mt1 = qc.filter_min_missing(mt1, 0.05)
#mt1 = qc.filter_max_maf(mt1, float(0.02)) # note, this filter removes all damaging variation. Don't use.
#mt1 = annotate_vep(mt1, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
mt1 = qc.get_table('data/mt/ukb_wes_200k_annotated_chr22.mt', 'mt')
#mt1 = mt1.annotate_rows(vep = mt1.vep.annotate(Gene = mt1.vep.worst_csq_by_gene_canonical.gene_id))
mt1 = analysis.filter_vep(mt1, 'consequence_category',['damaging_missense','ptv'])
mt1.count()

(7971, 199795)

In [64]:
#ht = mt1.explode_rows(mt1.vep.worst_csq_by_gene_canonical)
#mt1.vep.worst_csq_by_gene_canonical.gene_id.show()

#mt1.vep.worst_csq_by_gene_canonical.gene_id

In [5]:
#mt2 = qc.get_table('data/unphased/tmp/ukb_wes_200k_filtered_2samples_chr22.vcf.bgz', 'vcf')
#mt2 = qc.filter_max_mac(mt2, 1) # get singletons
#mt2 = qc.filter_min_missing(mt2, 0.05)
#mt2 = analysis.annotate_vep(mt2, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
#mt2 = analysis.filter_vep(mt2, 'consequence_category',['damaging_missense','ptv'])

In [6]:
#ko_prob = analysis.gene_csqs_calc_pKO(mt1, mt2)
#ko_prob = ko_prob.filter_entries(ko_prob.pKO > 0.9)
#ko_prob.show()
#ko_prob.show()

In [77]:
#mt_rs = gene_csqs_dosage_builder(mt1).entries()
#mt_rs = mt_rs.filter(hl.literal(set([2])).contains(mt_rs.DT))
#mt_rs.show()

2021-09-22 14:11:43 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-22 14:12:08 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-22 14:12:31 Hail: INFO: Ordering unsorted dataset with network shuffle


,,,
gene_id,s,eur,DT
str,str,int32,int32
"""ENSG00000025708""","""1000120""",1,2
"""ENSG00000025708""","""1000473""",1,2
"""ENSG00000025708""","""1001276""",1,2
"""ENSG00000025708""","""1002461""",1,2
"""ENSG00000025708""","""1002688""",1,2
"""ENSG00000025708""","""1003238""",1,2
"""ENSG00000025708""","""1003518""",1,2
"""ENSG00000025708""","""1005039""",0,2


In [73]:
# working
#mt_rs = gene_csqs_case_builder(mt1).entries()
#mt_rs = mt_rs.filter(hl.literal(set(['CH'])).contains(mt_rs.csqs))
#mt_rs.show()

2021-09-22 14:03:54 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-22 14:04:18 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-22 14:04:41 Hail: INFO: Ordering unsorted dataset with network shuffle


,,,
gene_id,s,eur,csqs
str,str,int32,str
"""ENSG00000025708""","""1019379""",1,"""CH"""
"""ENSG00000025708""","""1044723""",1,"""CH"""
"""ENSG00000025708""","""1084256""",1,"""CH"""
"""ENSG00000025708""","""1185296""",1,"""CH"""
"""ENSG00000025708""","""1196627""",0,"""CH"""
"""ENSG00000025708""","""1216210""",1,"""CH"""
"""ENSG00000025708""","""1249714""",0,"""CH"""
"""ENSG00000025708""","""1303426""",1,"""CH"""


In [5]:
#rs = gene_csqs_variant_builder(mt1)
#rs.show()

In [106]:
# load file
mt = qc.get_table('data/mt/ukb_wes_200k_annotated_chr22.mt', 'mt')
mt = analysis.filter_vep(mt, 'consequence_category',['damaging_missense','ptv'])

def gene_strand_builder(mt, field = 'snpid'):
    '''Returns hail table that contains genes, samples, rsids, knockout status'''
    
    # annotate entries with phased data
    mt = mt.explode_rows(mt.vep.worst_csq_by_gene_canonical)
    mt = mt.annotate_entries(rsid_entry = mt[field])
    mt = mt.annotate_rows(gene_id = mt.vep.worst_csq_by_gene_canonical.gene_id)
    mt = analysis.annotate_phased_entries(mt)

    # filter to each strand
    strand1 = mt.filter_entries((mt.a0_alt == True) | (mt.a_homo == True)).entries() # sets entries to NA in matrix table
    strand2 = mt.filter_entries((mt.a1_alt == True) | (mt.a_homo == True)).entries()

    # filter to each gene
    strand1 = strand1.group_by(strand1.gene_id, strand1.s).aggregate(phase1 = hl.agg.collect(strand1.rsid_entry))
    strand2 = strand2.group_by(strand2.gene_id, strand2.s).aggregate(phase2 = hl.agg.collect(strand2.rsid_entry))

    # combine each strand
    ht = strand1.annotate(phase2 = strand2[strand1.gene_id, strand1.s].phase2)
    ht = ht.annotate(knockout = (~hl.is_missing(ht.phase1)) & ~(hl.is_missing(ht.phase2)))
    return ht

def gene_csqs_knockout_builder(in_mt, keep = None):
    '''Return a hail table that contains knockout status alongside phase of variants in genes'''
    mt_rs = gene_strand_builder(in_mt, 'snpid')
    mt_dt = analysis.gene_csqs_case_builder(in_mt)
    combined = mt_rs.annotate(csqs = mt_dt[mt_rs.gene_id, mt_rs.s].csqs)
    if keep is not None:
        combined = combined.filter(hl.literal(keep).contains(combined.csqs))
    return combined




In [110]:
#gene_strand_snpid_builder(mt1).describe()
#analysis.gene_csqs_case_builder(mt).describe()
res = gene_csqs_knockout_builder(mt, keep = ['CH'])
res.show()

2021-09-22 18:07:34 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-22 18:08:18 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-22 18:09:09 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-22 18:09:54 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-22 18:10:42 Hail: INFO: Ordering unsorted dataset with network shuffle


,,,,,
gene_id,s,phase1,phase2,knockout,csqs
str,str,array<str>,array<str>,bool,str
"""ENSG00000025708""","""1019379""","[""chr22_50525778_G_A""]","[""chr22_50522253_C_T""]",True,"""CH"""
"""ENSG00000025708""","""1044723""","[""chr22_50522253_C_T""]","[""chr22_50525778_G_A""]",True,"""CH"""
"""ENSG00000025708""","""1084256""","[""chr22_50524144_G_A""]","[""chr22_50522253_C_T""]",True,"""CH"""
"""ENSG00000025708""","""1185296""","[""chr22_50524011_G_T""]","[""chr22_50522253_C_T""]",True,"""CH"""
"""ENSG00000025708""","""1196627""","[""chr22_50529755_C_A""]","[""chr22_50522253_C_T""]",True,"""CH"""
"""ENSG00000025708""","""1216210""","[""chr22_50525778_G_A""]","[""chr22_50522253_C_T""]",True,"""CH"""
"""ENSG00000025708""","""1249714""","[""chr22_50522253_C_T""]","[""chr22_50524168_T_C""]",True,"""CH"""
"""ENSG00000025708""","""1303426""","[""chr22_50522253_C_T""]","[""chr22_50523994_C_T""]",True,"""CH"""


In [133]:
# input path
#vep_path = args.vep_path
#input_path = args.input_path
#input_type = args.input_type
#out_prefix = args.out_prefix
#out_type = args.out_type
#translate_iid = args.translate_iid
    
# annotate with VEP
#hail_init.hail_bmrc_init('/logs/hail/hail_vep_export.log', 'GRCh38')
#dataset = qc.get_table(input_path, input_type)
dataset = qc.get_table('data/mt/ukb_wes_200k_annotated_chr22.mt', 'mt')
vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf'    

# clean up snpID and rsID
dataset = qc.annotate_snpid(dataset)
dataset = qc.annotate_rsid(dataset)
dataset = qc.default_to_snpid_when_missing_rsid(dataset)

# recalc info
#dataset = qc.recalc_info(dataset)

# get VEP
result = hl.vep(dataset, "utils/configs/vep_env.json") 
result = process_consequences(result)
result = analysis.annotate_dbnsfp(result, vep_path)
result = analysis.variant_csqs_category_builder(result)

# export as row format
result = result.explode_rows(result.vep.worst_csq_by_gene_canonical)
result = result.annotate_rows(vep = result.vep.worst_csq_by_gene_canonical)
result = result.drop('vep_proc_id')
result = result.rows()
result.export(out_prefix + 'variants.tsv.gz')

2021-09-22 18:21:17 Hail: WARN: expected input file 'file:/well/lindgren/flassen/ressources/dbsnp/GRCh38/GCF_000001405.39.gz' to end in .vcf[.bgz, .gz]


Annotating with VEP file: data/vep/full/ukb_wes_200k_full_vep_chr22.vcf


----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'rsid': str 
    'qual': float64 
    'filters': set<str> 
    'info': struct {
        AF: float64, 
        AC: int32, 
        CM: array<float64>, 
        AN: int32
    } 
    'vep': struct {
        allele_num: int32, 
        amino_acids: str, 
        biotype: str, 
        canonical: int32, 
        ccds: str, 
        cdna_start: int32, 
        cdna_end: int32, 
        cds_end: int32, 
        cds_start: int32, 
        codons: str, 
        consequence_terms: array<str>, 
        distance: int32, 
        domains: array<struct {
            db: str, 
            name: str
        }>, 
        exon: str, 
        gene_id: str, 
        gene_pheno: int32, 
        gene_symbol: str, 
        gene_symbol_source: str, 
        hgnc_id: str, 
        hgvsc: str, 
        hgvsp: str, 
        hgvs_offset: i

In [47]:
mt.vep.Gene.show()

2021-09-20 21:00:32 Hail: INFO: Coerced almost-sorted dataset
2021-09-20 21:00:34 Hail: INFO: Coerced sorted dataset


,,
locus,alleles,
locus<GRCh38>,array<str>,str
chr22:15528363,"[""C"",""T""]","""ENSG00000130538"""
chr22:15528649,"[""G"",""A""]","""ENSG00000130538"""
chr22:15528650,"[""G"",""A""]","""ENSG00000130538"""
chr22:15528668,"[""G"",""A""]","""ENSG00000130538"""
chr22:15528753,"[""C"",""T""]","""ENSG00000130538"""
chr22:16591038,"[""G"",""A""]","""ENSG00000198445"""
chr22:16591083,"[""G"",""A""]","""ENSG00000198445"""
chr22:16591104,"[""C"",""A""]","""ENSG00000198445"""


In [19]:
gene_csqs_vep_builder(mt1).describe()
    #combined.entries().show()
#mt1.describe()
#res = gene_csqs_dosage_builder(mt1)
#res.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'Gene': str
----------------------------------------
Entry fields:
    's0': array<str>
    's1': array<str>
    'hom_var': array<str>
----------------------------------------
Column key: ['s']
Row key: ['Gene']
----------------------------------------


In [78]:
res = gene_csqs_dosage_builder(mt1)
res.show()

2021-09-17 16:15:24 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:15:33 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:15:36 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:15:40 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:15:42 Hail: INFO: Coerced sorted dataset
2021-09-17 16:17:22 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-17 16:17:22 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:17:23 Hail: INFO: Coerced sorted dataset
2021-09-17 16:19:04 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-17 16:19:05 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:19:06 Hail: INFO: Coerced sorted dataset
2021-09-17 16:20:46 Hail: INFO: Ordering unsorted dataset with network shuffle


,,
,'1705888','3544034'
Gene,DT,DT
str,int32,int32
"""ENSG00000008735""",0,0
"""ENSG00000015475""",0,0
"""ENSG00000025770""",0,0
"""ENSG00000040608""",0,0
"""ENSG00000054611""",0,0
"""ENSG00000056487""",0,0
"""ENSG00000063515""",0,0


In [43]:
res.filter(hl.literal(['CH','HO']).contains(res.gene_csqs)).show()

2021-09-17 15:53:59 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 15:54:01 Hail: INFO: Coerced sorted dataset
2021-09-17 15:55:51 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-17 15:55:51 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 15:55:53 Hail: INFO: Coerced sorted dataset
2021-09-17 15:57:39 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-17 15:57:40 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 15:57:42 Hail: INFO: Coerced sorted dataset
2021-09-17 15:59:27 Hail: INFO: Ordering unsorted dataset with network shuffle


,,,,,
Gene,s,s0,s1,hom_var,gene_csqs
str,str,bool,bool,bool,str
"""ENSG00000133454""","""1705888""",True,True,False,"""CH"""
"""ENSG00000184113""","""1705888""",False,False,True,"""HO"""
"""ENSG00000215568""","""3544034""",False,False,True,"""HO"""


In [17]:
res.show()

2021-09-17 15:36:30 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 15:36:32 Hail: INFO: Coerced sorted dataset
2021-09-17 15:38:17 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-17 15:38:18 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 15:38:19 Hail: INFO: Coerced sorted dataset
2021-09-17 15:40:04 Hail: INFO: Ordering unsorted dataset with network shuffle


,,,
Gene,s,s0,s1
str,str,bool,bool
"""ENSG00000133454""","""1705888""",True,True


In [17]:
#res = gene_csqs_dosage_builder(mt1)
#res.filter(hl.literal(['CH','HO']).contains(res.gene_csqs)).show()
#res.show()
res = analysis.gene_csqs_rsid_builder(mt1).entries()
res.filter(hl.literal(['CH','HO']).contains(res.gene_csqs)).show()

AttributeError: Table instance has no field, method, or property 'gene_csqs'
    Did you mean:
        Data fields: 'Gene' [row], 'csqs' [row]

In [76]:
ht0_vars.show()

2021-09-17 16:12:06 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:12:08 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:12:10 Hail: INFO: Coerced sorted dataset
2021-09-17 16:13:50 Hail: INFO: Ordering unsorted dataset with network shuffle


,,
,'1705888','3544034'
Gene,rsid,rsid
str,array<str>,array<str>
"""ENSG00000008735""",[],[]
"""ENSG00000015475""",[],[]
"""ENSG00000025770""",[],[]
"""ENSG00000040608""",[],[]
"""ENSG00000054611""",[],[]
"""ENSG00000056487""",[],[]
"""ENSG00000063515""",[],[]


In [ ]:
def calc_p_ko(mt):
    '''Annotates entries with P(Knockout). Requires, that fields are
       already annotated with "singletons" count, and "DT". '''
    ko_mt = mt.annotate_entries(
        pKO = hl.if_else(
            mt.DT == 2, 1, # knockout
            hl.if_else(
                mt.DT == 1, 
                hl.if_else(mt.singletons >= 1, 1 - (1/2)**mt.singletons, 0), # one phased hetz
                hl.if_else(mt.singletons >= 2, 1 - 2*(1/2)**mt.singletons, 0), # zero phased hetz
            )
        )
    )
    return ko_mt

In [6]:
def gene_csqs_calc_pKO(mt_phased, mt_unphased, fields_drop = ['dosage','sigletons']):
    
    '''
    Annotates entries with P(Knockout). Requires a phased matrix and an unphased
    matrix that only contains singletons.
    '''
    
    # setup variables
    mt1 = mt_phased # contains phased non-singletons
    mt2 = mt_unphased # contains unphased singletons
    
    # Determine probability of being KO given singletons and phased hetz
    mt1_burden = gene_csqs_dosage_builder(mt1)
    mt2_burden = gene_burden_annotations_per_sample(mt2)
    mt_ko = mt1_burden.annotate_entries(singletons = mt2_burden[(mt1_burden.Gene, mt1_burden.s)].n)
    mt_ko = mt_ko.annotate_entries(singletons = hl.if_else(~hl.is_missing(mt_ko.singletons), mt_ko.singletons, 0 ))
    mt_ko = calc_p_ko(mt_ko)

    # drop not needed rows
    mt_ko = mt_ko.drop(*[f for f in fields_drop if f in mt_ko.entry])
    #mt_ko_entries = mt_ko.entries()
    #mt_ko_entries = mt_ko_entries.filter(~hl.is_missing(mt_ko_entries.pKO))
    return mt_ko

In [7]:
def gene_csqs_calc_pKO_pseudoSNP(mt1, mt2, chrom):
    '''Calculate probability of being a knockout incoporating phased 
       data (mt1) and unphased singletons (mt2). Create a file with 
       fake markers, that can be inputted into SAIGE'''
    # mt1 is phased
    # mt2 is unphased

    # get probability matrix
    pmt = get_prob_ko_matrix(mt1, mt2, ["DT","singletons"])
    pmt = pmt.annotate_entries(DT = pmt.pKO*2) # multiply probability by 2 (dosage encoded [0:2])
    pmt = pmt.drop('pKO')

    # create fake loci
    pmt = pmt.annotate_rows(locus = hl.parse_locus('chr' + str(chrom) + ':1'))
    pmt = pmt.annotate_rows(alleles = hl.literal(['X','Y']))
    pmt = pmt.annotate_rows(rsid = pmt.Gene)
    pmt = pmt.key_rows_by(pmt.locus, pmt.alleles)
    pmt = pmt.drop('Gene')
    return pmt

In [79]:
#get_prob_ko_matrix(mt1, mt2)
analysis.construct_phased_dosage_mt(mt1).show()

2021-09-17 16:21:40 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:21:42 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:21:43 Hail: INFO: Coerced sorted dataset
2021-09-17 16:23:21 Hail: INFO: Ordering unsorted dataset with network shuffle


,,
,'1705888','3544034'
Gene,dosage,dosage
str,int32,int32
"""ENSG00000008735""",0,0
"""ENSG00000015475""",0,0
"""ENSG00000025770""",0,0
"""ENSG00000040608""",0,0
"""ENSG00000054611""",0,0
"""ENSG00000056487""",0,0
"""ENSG00000063515""",0,0


In [80]:
analysis.construct_phased_dosage_mt(mt1).describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'Gene': str
----------------------------------------
Entry fields:
    'dosage': int32
----------------------------------------
Column key: ['s']
Row key: ['Gene']
----------------------------------------


In [97]:
def is_phased(mt):
    ''' Check if the input contains phased data. Returns Bool'''
    mt = mt.annotate_entries(phased = 
                        (mt.GT ==  hl.parse_call('0|0')) |
                        (mt.GT ==  hl.parse_call('1|0')) |
                        (mt.GT ==  hl.parse_call('0|1')) |
                        (mt.GT ==  hl.parse_call('1|1'))
                       )
    return mt.aggregate_entries(hl.agg.any(mt.phased))



In [98]:
is_phased(mt1)

2021-09-17 16:46:27 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 16:46:29 Hail: INFO: Coerced sorted dataset


True

In [105]:
def maf_category_case_builder(mt):
    return (hl.case()
            .when(call_stats_expr.AF <= 0.00001, 0.00001)
            .when(call_stats_expr.AF <= 0.0001, 0.0001)
            .when(call_stats_expr.AF <= 0.001, 0.001)
            .when(call_stats_expr.AF <= 0.01, 0.01)
            .when(call_stats_expr.AF <= 0.1, 0.1)
            .default(0.99))

def mac_category_case_builder(call_stats_expr):
    return (hl.case()
            .when(call_stats_expr.AC <= 5, call_stats_expr.AC)
            .when(call_stats_expr.AC <= 10, 10)
            .when(call_stats_expr.AC <= 25, 25)
            .when(call_stats_expr.AC <= 100, 100)
            .when(call_stats_expr.AC <= 1000, 1000)
            .when(call_stats_expr.AC <= 10000, 10000)
            .when(call_stats_expr.AC <= 100000, 100000)
            .when(call_stats_expr.AC <= 1000000, 1000000)
            .default(0))

In [107]:
mac_category_case_builder(mt.info).show()

2021-09-17 17:28:26 Hail: INFO: Coerced almost-sorted dataset
2021-09-17 17:28:27 Hail: INFO: Coerced sorted dataset


,,
locus,alleles,
locus<GRCh38>,array<str>,int32
chr22:15528363,"[""C"",""T""]",0
chr22:15528649,"[""G"",""A""]",0
chr22:15528650,"[""G"",""A""]",0
chr22:15528668,"[""G"",""A""]",0
chr22:15528753,"[""C"",""T""]",0
chr22:16591038,"[""G"",""A""]",0
chr22:16591083,"[""G"",""A""]",0
chr22:16591104,"[""C"",""A""]",0
